# 3D Resist Depth Dose (Phase 5)

Tier 1 matplotlib 3D view of dose attenuation through resist depth.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from src.mask import MaskGrid, kirchhoff_mask, line_space_pattern
from src.pupil import PupilSpec
from src.aerial import aerial_image
from src.resist_depth import depth_resolved_dose_stack

In [ ]:
grid = MaskGrid(nx=256, ny=64, pixel_size=2e-9)
mask = kirchhoff_mask(line_space_pattern(grid, pitch_m=40e-9))
aerial, wafer = aerial_image(mask, grid, pupil_spec=PupilSpec(grid_size=256), anamorphic=False)
depths = np.linspace(0.0, 60e-9, 7)
dose_stack = depth_resolved_dose_stack(aerial, depths, dose=1.0, absorption_coefficient_m_inv=2.0e6)
print('dose stack shape:', dose_stack.shape)

In [ ]:
x_nm = (np.arange(wafer.nx) - wafer.nx / 2) * wafer.pixel_x_m * 1e9
X, Z = np.meshgrid(x_nm, depths * 1e9)
Y = dose_stack[:, wafer.ny // 2, :]

fig = plt.figure(figsize=(9, 5))
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(X, Z, Y, cmap='magma', linewidth=0, antialiased=True, alpha=0.9)
ax.set_xlabel('x [nm]')
ax.set_ylabel('depth [nm]')
ax.set_zlabel('dose')
ax.set_title('Depth-resolved resist dose')
fig.tight_layout()
plt.show()